In [ ]:
!pip install opencv-python tensorflow matplotlib

In [ ]:
import os
import cv2
import zipfile
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.model_selection import train_test_split

from google.colab import files

In [ ]:
uploaded = files.upload()
zip_path = list(uploaded.keys())[0]

extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted!")

TypeError: 'NoneType' object is not subscriptable

In [ ]:
base = extract_path

while len(os.listdir(base)) == 1:
    base = os.path.join(base, os.listdir(base)[0])

dataset_path = base
classes = os.listdir(dataset_path)

print("Classes:", classes)

In [ ]:
data = []
labels = []

for label, cls in enumerate(classes):
    path = os.path.join(dataset_path, cls)

    for img in os.listdir(path)[:500]:
        img_path = os.path.join(path, img)

        image = cv2.imread(img_path)
        if image is None:
            continue

        image = cv2.resize(image, (64,64))
        data.append(image)
        labels.append(label)

X = np.array(data) / 255.0
y = np.array(labels)

print("Total samples:", len(X))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),

    Dense(len(classes), activation='softmax')   # FIXED
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=10)

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

In [ ]:
def detect_cars(image):
    image = cv2.resize(image, (256,256))
    h, w = image.shape[:2]

    boxes = []

    for y in range(0, h-64, 32):
        for x in range(0, w-64, 32):
            patch = image[y:y+64, x:x+64]

            patch = patch / 255.0
            patch = np.expand_dims(patch, axis=0)

            pred = model.predict(patch, verbose=0)
            label = np.argmax(pred)
            confidence = np.max(pred)

            if confidence > 0.8 and classes[label].lower() == "car":
                boxes.append([x, y, 64, 64])

    if len(boxes) == 0:
        return image

    boxes = np.array(boxes)

    x_min = np.min(boxes[:,0])
    y_min = np.min(boxes[:,1])
    x_max = np.max(boxes[:,0] + boxes[:,2])
    y_max = np.max(boxes[:,1] + boxes[:,3])

    cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0,255,0), 3)

    return image

In [ ]:
uploaded = files.upload()

for file in uploaded.values():
    file_bytes = np.frombuffer(file, np.uint8)
    img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    result = detect_cars(img)

    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()